# 02. Visualization — Seaborn(정적) + Plotly(인터랙티브)

- Seaborn 정적 차트 1개 이상, Plotly 인터랙티브 차트 1개 이상
- 분포 / 상관관계 / 그룹 비교 중 선택해 표현 (차트 제목·축 레이블 필수)

In [ ]:
import sys

sys.path.append("..")

from pathlib import Path

import pandas as pd

from src import viz

In [ ]:
card = pd.read_parquet(Path("..") / "data" / "processed" / "card_clean.parquet")
card.shape

## 1. Seaborn 정적 차트 — 시간대별 팁 지급률/팁 비율 (그룹 비교)

In [ ]:
from IPython.display import Image

fig_dir = Path("..") / "reports" / "figures"
seaborn_path = viz.plot_tip_pct_distribution(card, fig_dir / "tip_by_hour_seaborn.png")
Image(filename=str(seaborn_path))

## 2. Plotly 인터랙티브 차트 — 거리구간 x 혼잡구역별 평균 팁 비율 (그룹 비교)

In [ ]:
import plotly.express as px

# viz.plot_tip_pct_interactive()는 파일로 저장하는 함수라, 노트북에서 바로 보려면
# 같은 집계를 다시 만들어 fig.show()로 인라인 표시한다.
plotly_path = viz.plot_tip_pct_interactive(card, fig_dir / "tip_by_distance_congestion_plotly.html")

df_plot = card.copy()
bins = [0, 1, 3, 5, 10, 20, 1000]
df_plot["distance_bin"] = pd.cut(df_plot["trip_distance"], bins=bins).astype(str)
df_plot["혼잡구역"] = df_plot["is_congestion"].map({True: "혼잡구역", False: "비혼잡구역"})
grouped = (
    df_plot.groupby(["distance_bin", "혼잡구역"], observed=True)["tip_pct"].mean().reset_index()
)

fig = px.bar(
    grouped,
    x="distance_bin",
    y="tip_pct",
    color="혼잡구역",
    barmode="group",
    title="거리구간 x 혼잡구역별 평균 팁 비율",
    labels={"distance_bin": "거리구간(마일)", "tip_pct": "평균 팁 비율"},
)
fig.show()